<a href="https://colab.research.google.com/github/AYUSHDHANGAR/RAG/blob/main/AYUSH_resume_pdf_RAG_implementation_using_gemini_api_%26_gemini_embedding_with_FAISS_vector_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -U langchain langchain-core
!pip install langchain-classic

In [ ]:
!pip install -U langchain-google-genai
!pip install -U langchain-community faiss-cpu
!pip install langchain_text_splitters
!pip install langchain openai tiktoken rapidocr-onnxruntime

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [ ]:
from google.colab import userdata
import os

GOOGLE_API_KEY = userdata.get("gemini_api_key")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [ ]:
!pip install pypdf
import os
import requests
import pandas as pd
from pypdf import PdfReader

import re
import random

In [ ]:
def text_formatter(text: str) -> str:
    """Clean and format extracted text."""

    cleaned_text = text.replace("\n", " ").strip()

    # Remove multiple spaces
    cleaned_text = re.sub(r"\s+", " ", cleaned_text)

    return cleaned_text

In [ ]:

pdf_path= "/content/Ayush_Resume_searchable.pdf"


reader = PdfReader(pdf_path)

print("Pages:", len(reader.pages))

page = reader.pages[0]

text = page.extract_text()

text = text_formatter(text)


print("\n--- Full Page Text ---\n")
print(text)

Pages: 1

--- Full Page Text ---

Ayush Dhangar ayushdhangar12345@gmail.com | +91 9528702189 | Mathura CAREER OBJECTIVE Computer Science (Al & ML) student seeking an entry-level Software Developer, Al/ML Engineer, or Full-Stack Developer role to apply technical skills, contribute to innovative projects, and grow in a dynamic technology-driven environment. WORK EXPERIENCE Machine Learning ¢ Internship Jun 2025 - Dec 2025 Pantech.ai, Virtual Completed an Al & Machine Learning internship at Pantech Al, where | learned core ML concepts, worked on multiple machine learning projects, and gained practical experience in data analysis, model building, and Al application development. EDUCATION B.Tech, Computer Science & Engineering 2024 - 2027 G. L. Bajaj Group Of Institutions, Mathura Diploma, Electrical Engineering 2022 - 2024 P.M.V. Polytechnic Percentage: 70.00% Secondary (X), CBSE 2024 Mount Hill Academy, Mathura Percentage: 75.00% TRAININGS / CERTIFICATIONS PORTFOLIO GitHub link 7 Python (

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=20
)

chunks = text_splitter.split_text(text)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Number of chunks: 8

--- Chunk 1 ---
Ayush Dhangar ayushdhangar12345@gmail.com | +91 9528702189 | Mathura CAREER OBJECTIVE Computer Science (Al & ML) student seeking an entry-level Software Developer, Al/ML Engineer, or Full-Stack Developer role to apply technical skills, contribute to innovative projects, and grow in a dynamic technology-driven environment. WORK EXPERIENCE Machine Learning ¢ Internship Jun 2025 - Dec 2025

--- Chunk 2 ---
Jun 2025 - Dec 2025 Pantech.ai, Virtual Completed an Al & Machine Learning internship at Pantech Al, where | learned core ML concepts, worked on multiple machine learning projects, and gained practical experience in data analysis, model building, and Al application development. EDUCATION B.Tech, Computer Science & Engineering 2024 - 2027 G. L. Bajaj Group Of Institutions, Mathura Diploma,

--- Chunk 3 ---
Mathura Diploma, Electrical Engineering 2022 - 2024 P.M.V. Polytechnic Percentage: 70.00% Secondary (X), CBSE 2024 Mount Hill Academy, Mathura Perc

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

In [ ]:
chunks

['Ayush Dhangar ayushdhangar12345@gmail.com | +91 9528702189 | Mathura CAREER OBJECTIVE Computer Science (Al & ML) student seeking an entry-level Software Developer, Al/ML Engineer, or Full-Stack Developer role to apply technical skills, contribute to innovative projects, and grow in a dynamic technology-driven environment. WORK EXPERIENCE Machine Learning ¢ Internship Jun 2025 - Dec 2025',
 'Jun 2025 - Dec 2025 Pantech.ai, Virtual Completed an Al & Machine Learning internship at Pantech Al, where | learned core ML concepts, worked on multiple machine learning projects, and gained practical experience in data analysis, model building, and Al application development. EDUCATION B.Tech, Computer Science & Engineering 2024 - 2027 G. L. Bajaj Group Of Institutions, Mathura Diploma,',
 'Mathura Diploma, Electrical Engineering 2022 - 2024 P.M.V. Polytechnic Percentage: 70.00% Secondary (X), CBSE 2024 Mount Hill Academy, Mathura Percentage: 75.00% TRAININGS / CERTIFICATIONS PORTFOLIO GitHub li

In [ ]:
vectors = embeddings.embed_documents(chunks)

print("Number of chunks:", len(chunks))
print("Number of vectors:", len(vectors))
print("Vector dimension:", len(vectors[0]))

Number of chunks: 8
Number of vectors: 8
Vector dimension: 3072


In [ ]:
print(vectors[0][:10])

[-0.0036631399, 0.000939354, 0.005619605, -0.056560725, -0.03136939, 0.0013984912, 0.029313074, 0.006302187, -0.014116285, -0.005531884]


In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_texts(
    texts=chunks,
    embedding=embeddings
)

print("FAISS vector store created!")

FAISS vector store created!


In [ ]:
query = "internship done by ayush"

results = vectorstore.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print(doc.page_content)


--- Result 1 ---
Ayush Dhangar ayushdhangar12345@gmail.com | +91 9528702189 | Mathura CAREER OBJECTIVE Computer Science (Al & ML) student seeking an entry-level Software Developer, Al/ML Engineer, or Full-Stack Developer role to apply technical skills, contribute to innovative projects, and grow in a dynamic technology-driven environment. WORK EXPERIENCE Machine Learning ¢ Internship Jun 2025 - Dec 2025

--- Result 2 ---
Jun 2025 - Dec 2025 Pantech.ai, Virtual Completed an Al & Machine Learning internship at Pantech Al, where | learned core ML concepts, worked on multiple machine learning projects, and gained practical experience in data analysis, model building, and Al application development. EDUCATION B.Tech, Computer Science & Engineering 2024 - 2027 G. L. Bajaj Group Of Institutions, Mathura Diploma,

--- Result 3 ---
Mathura Diploma, Electrical Engineering 2022 - 2024 P.M.V. Polytechnic Percentage: 70.00% Secondary (X), CBSE 2024 Mount Hill Academy, Mathura Percentage: 75.00% TR

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. LLM Initialize करें (Updated Model Name)
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.3
)

# 2. Retriever और Prompt Template
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

template = """आप एक सहायक AI हैं। नीचे दिए गए Context के आधार पर ही प्रश्न का उत्तर दें।
यदि उत्तर Context में नहीं है, तो स्पष्ट रूप से कहें कि जानकारी उपलब्ध नहीं है।

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 4. Response चेक करें
query = "qualification of ayush?"
response = rag_chain.invoke(query)

print("--- Answer ---")
print(response)

/usr/local/lib/python3.12/dist-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


--- Answer ---
Context के अनुसार, आयुष की योग्यताएँ (Qualifications) निम्नलिखित हैं:

1. **कंप्यूटर साइंस (AI & ML):** वह वर्तमान में कंप्यूटर साइंस (AI & ML) के छात्र हैं।
2. **डिप्लोमा (इलेक्ट्रिकल इंजीनियरिंग):** P.M.V. पॉलिटेक्निक से (2022 - 2024) - 70.00% अंक।
3. **सेकेंडरी (10वीं):** माउंट हिल एकेडमी, मथुरा (CBSE, 2024) - 75.00% अंक।
